# 05a. Multi-Agent 선택 기준 ①: 실패를 먼저 분류하고, 나누는 기준을 가른다

Agent를 몇 개로 나눌지는 이 노트북의 질문이 아닙니다. **나누기 전에 무엇을 고쳤고, 나눈 뒤 무엇이 좋아졌으며, 그 대가가 얼마인지를 자기 실행 표로 말할 수 있는가**가 질문입니다. 05는 두 권으로 나뉩니다. 이 노트북 **05a**는 Single의 실패를 분류하고 먼저 고친 뒤, 구조를 나누는 이득과 비용을 재고 나누는 기준(Persona, Context 경계, Skill, SubAgent)을 가릅니다. 수치 읽기, Fast/Slow Path, 같은 조건 비교표, 선택 기록은 **05b**에서 이어집니다.

이 노트북은 앞 노트북의 kernel 상태에 기대지 않고 단독으로 돕니다.

## 학습 목표

- 먼저 작업을 Workflow와 Agent로 나누고, Multi-Agent를 따질 대상이 모델이 맡은 칸뿐임을 확인합니다.
- 나누기 전에 Single의 실패를 분류하고, 프롬프트·도구·검색으로 고칠 문제를 먼저 고칩니다.
- 같은 사례에서 이득과 비용을 함께 재고, 조정 비용을 숫자로 말합니다.
- Persona 분리와 Context 분리를 실행으로 구분합니다.
- 코드, Skill, SubAgent 중 무엇을 쓸지 기준을 말합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 0 | 먼저 작업을 Workflow와 Agent로 나눔 | 4 |
| 1 | 먼저 Single의 실패를 분류 | 7 |
| 2 | 이득과 비용 | 5 |
| 3 | Context 경계 vs Persona | 5 |
| 4 | Skill vs SubAgent | 5 |
| 5 | 직접 해보기와 정리 | 4 |

## 호출 규모

**이 과정에서 비싼 노트북입니다.** 같은 사례를 여러 구조로 돌리기 때문입니다. 아래는 실행 전 예고 상한이고, 각 유료 셀이 실행 직전에 규모를 한 줄로 알립니다. 수강생 본인의 크레딧을 씁니다.

| 범위 | 모델 호출(예고 상한) | EXA |
|---|---:|---:|
| 1절 Single 기준선 4사례 + 프롬프트 한 줄 수정 | 20 + 8 | 0회 |
| 2절 Supervisor 같은 사례 | 20 | 0회 |
| 3절 Persona만 나눈 Worker | 20 | 0회 |
| 4절 `single_skills` 같은 사례 | 12 | 0회 |
| 기본 실행 합계 | 예고 상한 80 | 0회 |
| 직접 해보기의 `RUN_EXERCISE = True` | 해보기 1이 48, 해보기 2가 24 | 0회 |

`RUN_EXERCISE`는 기본값이 `False`라 전체 실행에 비용을 더하지 않습니다. 시간 여유와 크레딧을 보고 직접 켜세요. 호출 수는 실행마다 달라집니다.

In [ ]:
import inspect
import json
import time
from collections import Counter, defaultdict
from uuid import uuid4

from deepagents import create_deep_agent
from langchain.agents.middleware import wrap_model_call, wrap_tool_call
from langchain.agents.structured_output import ProviderStrategy
from langchain_core.messages import ToolMessage

from mas_lab.contracts import FinalAnswer, validate_final
from mas_lab.evaluation import evaluate_row, load_gold
from mas_lab.hybrid import build_hybrid
from mas_lab.runner import FIXED_BUDGET, make_desk, run_case, run_grid
from mas_lab.structures import OUTCOME_GUIDE, build_supervisor, gate_answer, scope_line
from mas_lab.trace import ModelMeter
from mas_lab.view import message_table, show_table
from mas_lab.workers import ALL_TOOLS, DENY_WRITES, ROLE_TOOLS, allowlist_middleware, subagent_spec
from rag_lab.corpus import load_cases
from rag_lab.evidence import BudgetExceeded
from rag_lab.settings import ROOT, announce, key_status, make_model, require_key

settings = require_key()  # 키가 없으면 여기서 멈추고 .env에 채우는 방법을 알려 줍니다.
print(key_status(settings))
model = make_model()  # 객체를 만들 뿐 호출이 아닙니다.
cases = load_cases()
case_by_id = {case["id"]: case for case in cases}
gold = load_gold()  # 사후 평가에만 씁니다. 실행 경로의 프롬프트와 도구로 들어가지 않습니다.
print(f"사례 {len(cases)}개, 유형 {sorted({case['type'] for case in cases})}")

## 0. 먼저 작업을 Workflow와 Agent로 나눕니다
**문제.** 이 Part의 질문은 "Agent를 나눌까?"입니다. 그런데 이 질문은 작업을 먼저 Workflow와 Agent로 나타낸 뒤에야 뜻이 있습니다. 순서가 정해진 일, 규칙으로 판정할 수 있는 일, 사람이 승인해야 하는 일은 코드나 사람의 몫이고, 모델의 판단이 필요한 칸만 Agent의 몫입니다. 모델이 맡지 않은 칸을 여러 Agent로 나눈다고 판단이 더 좋아지지는 않습니다.

아래 셀은 04b 8절에서 조립한 hybrid 그래프를 **만들기만 하고 부르지 않습니다.** 그래프에서 노드 이름을 읽고, 노드마다 누가 맡는지(코드, 모델, 사람)를 표로 적습니다. 담당은 그래프 안에서 자동으로 알 수 없으므로 셀 안의 사전에 사람이 적고, 그래프에 있는 노드가 사전에 없으면 멈춥니다.

**먼저 예측하세요.** 노드가 열 개 남짓인 이 그래프에서 모델이 맡은 노드는 몇 개일까요? 나머지는 코드와 사람 중 누구의 몫일까요? 표를 보기 전에 노드 이름으로 적어 보세요.

In [ ]:
# [C] 모델을 부르지 않습니다. 그래프를 만들기만 하고 부르지 않습니다.

# 1단계: 사례 하나의 Desk를 만들고 hybrid 그래프를 만듭니다. 만드는 일은 모델 호출이 아닙니다.
split_desk = make_desk(case_by_id["complex-double-pay"], label="section0")
split_graph = build_hybrid(split_desk, model)

# 2단계: 컴파일된 그래프에서 노드 이름을 읽습니다. 시작과 끝 표지는 일이 아니라서 뺍니다.
graph_nodes = [name for name in split_graph.get_graph().nodes if not name.startswith("__")]

# 3단계: 노드마다 누가 맡는지와 이유를 적습니다. 코드를 읽고 사람이 정한 값입니다.
NODE_OWNER = {
    "scope": ("코드", "질문에 다른 고객 이름이 있는지 문자열로 확인합니다. 판단이 필요 없습니다."),
    "router": ("모델", "fast와 slow 중 어느 경로가 맞는지는 질문의 뜻을 읽어야 정할 수 있습니다."),
    "fast": ("모델", "조사 한두 번으로 고객에게 보낼 답을 씁니다."),
    "worker": ("모델", "결제, 규정, 장애 중 하나를 도구로 조사하고 사실을 정리합니다."),
    "retry": ("모델", "시스템 오류로 실패한 조사를 다시 돌립니다. 다시 돌 때는 Worker가 모델을 씁니다."),
    "join": ("코드", "Worker 결과를 모으고 빠진 것이 있는지 규칙으로 셉니다."),
    "writer": ("모델", "Worker 결과를 합쳐 고객에게 보낼 답을 씁니다."),
    "validate": ("코드", "인용 소속, 금액 출처, outcome 일관성을 규칙으로 검사합니다."),
    "gate": ("코드", "환불 정책을 코드로 판정합니다. 모델은 환불을 직접 실행하지 못합니다."),
    "approve": ("사람", "승인이 필요한 환불은 사람의 응답을 기다립니다 (interrupt)."),
    "execute": ("코드", "승인된 환불만 멱등 키로 원장에 씁니다."),
}

# 4단계: 그래프에는 있는데 사전에 없는 노드가 있으면 조용히 넘어가지 않고 멈춥니다.
missing_nodes = [name for name in graph_nodes if name not in NODE_OWNER]
if missing_nodes:
    raise ValueError(f"담당을 적지 않은 노드가 있습니다: {missing_nodes}. NODE_OWNER에 담당과 이유를 더하세요.")

# 5단계: 그래프의 노드 순서대로 표를 만들고, 담당별 개수를 함께 셉니다.
split_rows = [{"노드": name, "담당(코드/모델/사람)": NODE_OWNER[name][0], "이유": NODE_OWNER[name][1]} for name in graph_nodes]
show_table(split_rows)
owner_counts = Counter(row["담당(코드/모델/사람)"] for row in split_rows)
print("담당별 노드 수:", dict(owner_counts))
print("Multi-Agent를 따질 노드(담당이 모델):", [row["노드"] for row in split_rows if row["담당(코드/모델/사람)"] == "모델"])
print("모델 호출 0회")

**관찰할 것.** 표의 `담당` 칸을 위에서 아래로 읽고, 예측과 비교하세요.

- Multi-Agent를 나눌지 따지는 대상은 `담당`이 **모델**인 행뿐입니다. `scope`, `join`, `validate`, `gate`, `execute`처럼 코드가 맡은 행을 Agent 여러 개로 쪼개도 새로 생기는 판단이 없으므로 얻는 것이 없고 조정 비용만 듭니다.
- `approve`처럼 사람이 맡은 행은 Agent가 아니라 승인 흐름의 문제입니다. 04b 7절에서 다룬 `interrupt`가 그 자리입니다.
- 모델이 맡은 행 안에서도 `router`와 `fast`는 한 번의 판단이고, `worker`는 서로 다른 조사입니다. 05b의 7절까지 05가 재는 것은 "모델이 맡은 칸을 몇 개의 Agent로 나눌 때 이득과 비용이 어떻게 달라지는가"입니다.
- 담당을 바꾸면 질문도 바뀝니다. 예를 들어 `validate`를 모델에게 맡기는 쪽으로 바꾸면 그 칸이 새로 Multi-Agent를 따질 대상이 되지만, 규칙으로 정할 수 있는 검사를 모델에게 넘긴 이유를 먼저 말해야 합니다.

## 1. 먼저 Single의 실패를 분류
**문제.** "복잡하니까 Agent를 나누자"는 말은 대개 실패를 보기 전에 나옵니다. 그런데 Single이 틀린 이유가 검색어 하나, 도구 설명 한 줄이었다면 Agent를 셋으로 늘려도 같은 실패가 셋으로 복제될 뿐입니다. 나누기 전에 실패를 먼저 분류합니다.

아래 표는 **오류 통계가 아니라 진단 틀**입니다. "몇 %가 이 칸에 들어간다"고 말하는 자료가 아니라, 자기 실패를 어디서부터 점검할지 정하는 질문지입니다.

| 칸 | 어떻게 드러나는가 | 먼저 점검할 것 | 분리를 검토할 근거 |
|---|---|---|---|
| 정보가 부족함 | 필요한 원문이나 사실을 찾지 못하고 일반 지식으로 채우거나 비워 둠 | 검색어, 코퍼스, `not_found` 처리 지침 | 사실의 출처가 서로 달라 조사마다 다른 도구와 절차가 필요할 때 |
| 도구를 잘못 고름 | 엉뚱한 도구를 부르거나 필요한 도구를 건너뜀 | 도구 이름과 설명, 프롬프트의 지침 | 설명을 고쳐도 도구 수 때문에 계속 헷갈리고, 역할별 허용 목록이 실제로 선택지를 줄일 때 |
| Context가 너무 큼 | 호출을 거듭할수록 입력 토큰이 불어나고 앞의 조건을 놓침 | 도구 결과 크기, 필요한 원문만 읽기 | 줄여도 서로 무관한 조사의 원문이 한 Context에 남아 판단을 방해할 때 |
| 단계 의존성이 강함 | 앞 단계의 값 없이 뒤 단계를 수행하거나 순서를 어김 | 코드로 순서 고정(03의 순차 흐름) | 이 칸은 대개 **나눌 근거가 아닙니다.** 그래프로 순서를 고정하는 쪽이 먼저입니다 |

**먼저 예측하세요.** 아래 네 사례를 실행하기 전에, 각각 위 표의 어느 칸에 들어갈 실패가 날 것 같은지(또는 실패하지 않을지) 한 줄씩 적으세요.

- `complex-double-pay`: 결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요.
- `insufficient-unknown-payment`: 결제 번호 P-7777도 환불해 주세요.
- `scope-beta-from-alpha`: 베타 계약 기준으로 크레딧 비율을 알려 주세요.
- `complex-ops-log`: 타임아웃 점검 순서와 운영 접근 로그의 보관 기간을 함께 묻는 질문입니다.

### 그림으로 보기: 나누기 전에 실패를 분류하는 순서

위 표를 그림으로 줄였습니다. 실행한 표 두 개에서 실패를 읽고, 칸마다 먼저 점검할 것이 정해져 있습니다.

![Single 실행 뒤 자동 검사와 신호표를 읽어 정보 부족, 도구 선택, Context 크기, 단계 의존성 네 칸 중 어디인지 가리고 각 칸마다 먼저 점검할 것을 고친다. 자동 검사 문제이거나 실패가 아니면 분리의 근거가 아니다](./assets/05a-1-failure-first.png)

- 오른쪽 네 칸은 모두 "먼저 점검할 것"을 고친 뒤에도 남는 실패일 때만 분리를 검토합니다.
- 아래 칸은 분리 대상이 아닙니다. 표기만 다른 답이 자동 검사에서 떨어진 것은 Agent의 실패가 아닙니다.
- 단계 의존성은 코드로 순서를 고정하는 쪽이 먼저입니다.

In [ ]:
announce("Single 기준선 4사례", model_calls=20)
BASELINE_IDS = ["complex-double-pay", "insufficient-unknown-payment", "scope-beta-from-alpha", "complex-ops-log"]
single_rows = [run_case("single", case_by_id[case_id], model=model) for case_id in BASELINE_IDS]
RUN_COLUMNS = ["structure", "case_id", "status", "outcome", "model_calls", "tool_calls", "input_tokens", "output_tokens", "elapsed_seconds"]
show_table(single_rows, columns=RUN_COLUMNS)

In [ ]:
single_checked = [evaluate_row(row, gold) for row in single_rows]
CHECK_COLUMNS = ["case_id", "자동 검사", "기대 outcome", "실제 outcome", "누락 사실", "금지 포함", "누락 문서", "실패 칸"]
show_table(single_checked, columns=CHECK_COLUMNS)
print("자동 검사는 문자열 포함 검사입니다. '정답'이 아니므로 실패 행의 답을 직접 읽습니다.")
for row, check in zip(single_rows, single_checked, strict=True):
    if not check["자동 검사"]:
        print(f"\n[{row['case_id']}] 누락 사실 {check['누락 사실']}\n{row['answer']}")

In [ ]:
def signals(row):
    """판결이 아니라 신호입니다. 어느 칸인지는 사람이 고릅니다."""
    counts = Counter(row["tools_called"])
    missing = [doc for doc in gold[row["case_id"]]["required_docs"] if doc not in row["retrieved_docs"]]
    inputs = [call["input_tokens"] for call in row["calls"] if call["input_tokens"] is not None]
    return {"사례": row["case_id"], "도구를 불렀는가": bool(counts), "부른 도구": dict(counts),
            "필요한 문서를 찾았는가": not missing, "찾지 못한 문서": missing,
            "같은 도구 최대 반복": max(counts.values(), default=0),
            "첫 호출 입력 토큰": inputs[0] if inputs else None, "마지막 호출 입력 토큰": inputs[-1] if inputs else None,
            "입력 토큰 증가(배)": round(inputs[-1] / inputs[0], 1) if inputs and inputs[0] else None}

show_table([signals(row) for row in single_rows])
print("모델 호출 0회. 이미 끝난 실행의 기록만 읽습니다.")

In [ ]:
# 위 두 표를 보고 직접 적으세요. 칸은 정보가 부족함, 도구를 잘못 고름, Context가 너무 큼, 단계 의존성이 강함, 자동 검사 문제, 실패 아님 중 하나입니다.
DIAGNOSIS = {case_id: {"칸": "미판정", "근거": "미판정"} for case_id in BASELINE_IDS}
# DIAGNOSIS["complex-double-pay"] = {"칸": "...", "근거": "어느 표의 어느 칸을 보고 그렇게 판단했는지"}

show_table([{"사례": case_id, **entry} for case_id, entry in DIAGNOSIS.items()])
print(f"미판정 {sum(entry['칸'] == '미판정' for entry in DIAGNOSIS.values())}건")

**관찰할 것.** 두 표를 나란히 읽으세요. 자동 검사 표의 `실패 칸`이 어느 검사에서 떨어졌는지를 말하고, 신호표는 그 이유의 단서를 줍니다.

- `실패 칸`에 `필수 사실`이 있으면 **답 전문부터 읽으세요.** 표 아래에 실패 행의 답을 출력해 두었습니다. 같은 사실을 다른 표현으로 썼다면 Agent의 실패가 아니라 자동 검사의 한계입니다. 이때는 구조를 바꿔도, 프롬프트를 고쳐도 고칠 대상이 아닙니다.
- 실제로 사실이 빠졌고 `필요한 문서를 찾았는가`가 False면 먼저 **정보가 부족함** 칸을 의심합니다. 찾았는데도 사실이 빠졌다면 원문을 읽고도 쓰지 않은 것이니 지침 쪽입니다.
- `부른 도구`에 필요한 도구가 없으면 **도구를 잘못 고름** 칸입니다. 예를 들어 보상을 물었는데 `lookup_availability`가 없는 경우입니다.
- `입력 토큰 증가(배)`가 크고 앞에서 확인한 조건을 답에서 놓쳤다면 **Context가 너무 큼** 칸을 봅니다. 증가만으로 실패라고 부르지 않습니다.
- `실패 칸`이 비어 있는 사례도 있을 수 있습니다. 그 사례는 Single로 충분하다는 뜻이 아니라, **이번 1회 관찰에서 자동 검사가 잡지 못했다**는 뜻입니다.

모든 사례가 통과했다면 실패 근거 없이 나누려는 것이니, 나눌 이유를 실패가 아닌 다른 곳(병렬 지연, 권한 분리)에서 찾아야 합니다. 어느 쪽이든 `DIAGNOSIS`에 근거 칸의 이름을 적으세요. 이제 작업을 나누기 전에 고칠 수 있는 칸을 하나 골라 고쳐 봅니다.

In [ ]:
def run_custom(label, case, build):
    """run_case와 같은 예산, 게이트, 검증을 쓰고 조립만 바꿉니다. 원문 메시지까지 돌려줍니다. 실패도 행으로 남깁니다."""
    run_id, desk, meter = uuid4().hex, make_desk(case, label=label), ModelMeter()
    row = {"run_id": run_id, "structure": label, "case_id": case["id"], "type": case["type"], "repeat": 1, "status": "ok", "outcome": None,
           "errors": [], "answer": "", "uncertainty": [], "evidence_refs": [], "raw_messages": []}
    started = time.perf_counter()
    try:
        state = build(desk).invoke({"messages": [{"role": "user", "content": case["question"]}]}, config={"callbacks": [meter], "recursion_limit": 60})
        row["raw_messages"], final = state["messages"], state.get("structured_response")
        if final is None:
            row.update(status="validation_failed", errors=["최종 응답이 없습니다."])
        else:
            final = gate_answer(final, desk, request_id=run_id, question=case["question"])
            problems = validate_final(final, desk.known_refs(), desk.source_text())
            row.update(status="validation_failed" if problems else "ok", errors=problems, outcome=final.outcome, answer=final.text,
                       evidence_refs=final.evidence_refs, uncertainty=final.uncertainty)
    except BudgetExceeded as error:
        row.update(status="budget_exceeded", errors=[str(error)])
    except Exception as error:  # 노트북을 멈추지 않고 실패를 기록합니다.
        row.update(status="model_error", errors=[f"{type(error).__name__}: {error}"[:300]])
    totals, report = meter.totals(), desk.budget.report()
    row.update(model_calls=totals["model_calls"], tool_calls=report["counts"].get("tool", 0), input_tokens=totals["input_tokens"],
               output_tokens=totals["output_tokens"], usage_missing=totals["usage_missing"], elapsed_seconds=round(time.perf_counter() - started, 3),
               ledger_refunds=len(desk.ledger.refunds), calls=meter.calls, spans=desk.trace.spans, tools_called=[e["name"] for e in report["events"] if e["event"] == "tool"],
               denied=[e["name"] for e in report["events"] if e["event"] == "tool_denied"], retrieved_docs=sorted({e.doc_id for e in desk.context.evidence.values()}),
               cited_docs=sorted({e.doc_id for ref, e in desk.context.evidence.items() if ref in row["evidence_refs"]}))
    return row

print("run_custom", inspect.signature(run_custom), "| 고정 예산:", FIXED_BUDGET, "| 모델 호출 0회")

In [ ]:
announce("프롬프트 한 줄만 고쳐 다시 실행", model_calls=8)
PATCHES = {  # 정답 값을 넣지 않습니다. 행동 지침만 한 줄 고칩니다.
    "complex-double-pay": "요청이 여러 개 섞이면 요청마다 필요한 조회를 모두 마친 뒤 답하세요. 보상을 물으면 가용률 조회와 SLA·개별 약정 검색을 모두 거치세요.",
    "insufficient-unknown-payment": "조회 결과가 not_found이면 다른 결제로 바꿔 답하지 말고 outcome을 insufficient로 두어 조회되지 않았다고 쓰세요.",
    "scope-beta-from-alpha": "현재 범위가 아닌 고객의 계약을 물으면 outcome을 out_of_scope로 두고 그 계약의 내용을 단정하지 마세요.",
    "complex-ops-log": "질문에 주제가 둘 이상이면 주제마다 따로 검색하고 evidence_refs에 주제별 evidence_id를 모두 넣으세요."}
failed_ids = [check["case_id"] for check in single_checked if not check["자동 검사"]]
target_id = failed_ids[0] if failed_ids else BASELINE_IDS[0]
PROMPT_PATCH = PATCHES[target_id]

def build_patched_single(desk):
    return create_deep_agent(model=model, tools=desk.tools("all"), name="single",
                             system_prompt=f"고객 요청을 처리하는 Agent입니다. {scope_line(desk)}\n{OUTCOME_GUIDE}\n{PROMPT_PATCH}",
                             middleware=allowlist_middleware(desk.budget, ALL_TOOLS), permissions=DENY_WRITES,
                             response_format=ProviderStrategy(FinalAnswer))

print("대상 사례:", target_id, "(자동 검사 실패 사례)" if failed_ids else "(실패 사례가 없어 첫 사례로 변동만 봅니다)")
patched_row = run_custom("single+한 줄", case_by_id[target_id], build_patched_single)
print("상태:", patched_row["status"], "모델", patched_row["model_calls"], "회")

In [ ]:
before_row = next(row for row in single_rows if row["case_id"] == target_id)
after_check, before_check = evaluate_row(patched_row, gold), evaluate_row(before_row, gold)
compare = [{**{key: row[key] for key in ("structure", "status", "outcome", "model_calls", "input_tokens", "elapsed_seconds")},
            "도구": dict(Counter(row["tools_called"])), "자동 검사": check["자동 검사"], "실패 칸": check["실패 칸"], "누락 사실": check["누락 사실"]}
           for row, check in ((before_row, before_check), (patched_row, after_check))]
show_table(compare)
print("바꾼 줄:", PROMPT_PATCH)
print(f"\n[고치기 전 답]\n{before_row['answer']}\n\n[고친 뒤 답]\n{patched_row['answer']}\n")
print("바꾼 곳은 이 한 줄뿐입니다. 도구, 모델, 예산, 사례는 그대로이며 양쪽 모두 1회 관찰입니다. 모델 호출 0회.")

**관찰할 것.** 표의 두 행에서 `자동 검사`, `실패 칸`, `도구`를 위아래로 비교하세요.

- `실패 칸`이 줄었다면 먼저 **두 답을 나란히 읽으세요.** 도구 호출과 내용이 달라져서 통과했다면 이 문제는 **분리가 필요 없었던 문제**일 가능성이 큽니다. 한 줄로 고칠 수 있는 것을 조직도로 고치면 비용만 늘어납니다. 내용은 같고 표현만 달라졌는데 통과했다면, 고쳐진 것은 Agent가 아니라 자동 검사에 걸린 문자열입니다.
- 그대로이거나 다른 칸이 새로 떨어졌다면 다음 절의 근거가 됩니다. 다만 먼저 `부른 도구`가 달라졌는지 보세요. 지침이 행동을 바꾸지 못한 것과, 행동은 바뀌었는데 결과가 그대로인 것은 다른 문제입니다.
- 셀 출력의 `대상 사례`가 "실패 사례가 없어 첫 사례로 변동만 봅니다"라면 이 표는 프롬프트의 효과가 아니라 실행 사이의 변동을 보여 줍니다. 고친 줄의 공으로 읽지 마세요.
- 어느 쪽이든 각 1회입니다. 모델 결과는 호출마다 달라지므로 한 번 나아진 것을 "고쳐졌다"고 단정하지 않습니다. 나아졌다고 말하려면 같은 수정으로 반복해 보아야 하고, 그것이 해보기 1입니다.

다음 절은 이 사례를 포함한 복합 요청을 Supervisor로 돌렸을 때 무엇을 얻고 무엇을 내는지 잽니다.

## 2. 이득과 비용
**문제.** 구조를 바꾼 뒤 품질만 보거나 비용만 보면 판단이 틀어집니다. 품질이 나아졌어도 토큰과 시간이 함께 늘었을 수 있고, 품질이 같으면 늘어난 비용이 무엇을 산 것인지 답해야 합니다. Worker에게 내린 지시와 돌려받은 결과도 Supervisor의 입력이 되므로 **조정 비용**입니다. Single에는 이 비용이 없습니다.

위임 지시와 결과 전달의 크기를 재려면 원문 메시지가 필요합니다. 그래서 이 절은 `run_case`와 같은 예산, 게이트, 검증을 쓰면서 메시지를 함께 돌려주는 `run_custom`으로 `build_supervisor`를 실행합니다. 구조는 `run_case("supervisor", ...)`와 같습니다.

**먼저 예측하세요.** 같은 `complex-double-pay`를 Supervisor로 돌리면 입력 토큰은 Single의 몇 배가 될까요? 자동 검사는 나아질까요, 그대로일까요? 시간은 줄까요, 늘까요? 세 가지를 적고 넘어가세요.

In [ ]:
announce("Supervisor로 같은 사례", model_calls=20)
double_pay = case_by_id["complex-double-pay"]
supervisor_row = run_custom("supervisor", double_pay, lambda desk: build_supervisor(desk, model))
show_table([supervisor_row], columns=RUN_COLUMNS)
show_table(message_table(supervisor_row["raw_messages"]))

In [ ]:
def coordination_bytes(row):
    """task 호출의 인자가 위임 지시, task의 ToolMessage가 결과 전달입니다. 둘 다 Supervisor의 입력이 됩니다."""
    messages = row.get("raw_messages", [])
    orders = [call["args"] for message in messages for call in (getattr(message, "tool_calls", None) or []) if call["name"] == "task"]
    results = [str(message.content) for message in messages if isinstance(message, ToolMessage) and message.name == "task"]
    return {"위임 횟수": len(orders), "위임 지시 바이트": sum(len(json.dumps(order, ensure_ascii=False).encode()) for order in orders),
            "결과 전달 바이트": sum(len(text.encode()) for text in results)}

def side_by_side(row):
    check, first = evaluate_row(row, gold), next((call["input_tokens"] for call in row["calls"] if call["input_tokens"] is not None), None)
    return {"구조": row["structure"], "호출": row["model_calls"], "첫 호출 입력 토큰": first, "입력 토큰": row["input_tokens"], "출력 토큰": row["output_tokens"], "시간": row["elapsed_seconds"],
            "자동 검사": check["자동 검사"], "실패 칸": check["실패 칸"], "누락 사실": check["누락 사실"], **coordination_bytes(row)}

show_table([side_by_side(single_rows[0]), side_by_side(supervisor_row)])
print("Single은 위임이 없어 조정 칸이 0입니다. usage가 없는 호출은 입력 토큰에 더하지 않습니다. 모델 호출 0회.")

**관찰할 것.** 표의 `입력 토큰`, `시간`, `자동 검사`, `위임 지시 바이트`, `결과 전달 바이트`를 함께 읽으세요. 경우를 나누면 다음과 같습니다.

- **자동 검사가 Single 실패, Supervisor 통과**: 비용이 품질을 산 경우처럼 보입니다. 그렇게 말하기 전에 두 가지를 확인하세요. Single의 `누락 사실`이 정말 빠진 것인지 앞 절의 답 전문으로 확인합니다. 표현 차이였다면 품질 차이가 아닙니다. 그리고 앞 절에서 프롬프트 한 줄로도 고쳐졌다면 같은 품질을 더 싸게 얻을 수 있었던 것입니다.
- **둘 다 통과**: 품질이 같다면 Supervisor 쪽에서 더 든 비용은 무엇을 샀나요? 답할 수 있는 후보는 지연 단축, 역할별 도구 제한, 원문 격리입니다. 후보를 표의 칸으로 증명하지 못하면 비용만 는 것입니다. 늘어나는 칸은 `호출`, `시간`, 조정 바이트처럼 실행마다 다를 수 있으니 칸별로 따로 읽으세요. Single의 `입력 토큰`이 오히려 더 크게 나왔다면 Single이 이번에 같은 도구를 되풀이해 부른 것인지 앞 절 신호표의 `같은 도구 최대 반복`으로 확인하세요. 1회씩 잰 값이라 그 차이는 구조가 아니라 실행 변동일 수 있습니다.
- **Supervisor 실패**: `실패 칸`과 `누락 사실`을 보고, 위 메시지 표에서 Worker가 그 사실을 돌려줬는데 Supervisor가 놓쳤는지 찾으세요. 돌려받은 뒤 잃은 것은 전달 손실입니다.

`결과 전달 바이트`는 Worker 요약의 크기입니다. 이것이 작으면 Supervisor의 Context는 작지만 원문은 Worker에 남아 있어 나중에 다시 확인할 수 없습니다. 작을수록 좋다고 읽지 마세요. 그럼 Worker를 나누는 기준은 무엇일까요? 다음 절에서 역할 이름이 기준이 되는지 시험합니다.

## 3. Context 경계 vs Persona
**문제.** "결제 담당, 규정 담당, 장애 담당"이라는 이름을 붙이면 나눈 것처럼 보입니다. 그런데 이름이 다르다고 Worker가 보는 도구와 Context까지 달라지지는 않습니다. 앞 절의 Worker는 `ROLE_TOOLS`로 도구 범위가 서로 다른 Context 분리였습니다. 이번에는 **같은 프롬프트와 이름을 유지한 채** 도구만 모두 열어 Persona만 남긴 Worker를 돌립니다.

역할별 사용량은 Worker의 middleware에 칸을 하나 더해 잽니다. 도구 범위 말고는 Supervisor의 프롬프트, 사례, 예산을 그대로 둡니다.

02 3절에서 비교한 프레임워크가 자동으로 붙이는 `general-purpose` Worker도 같은 이야기입니다. 이 Worker는 부모와 같은 것을 보므로 Context 경계가 없고, 이름만 다른 Persona와 다르지 않습니다.

**먼저 예측하세요.** 이름만 다른 Worker 셋에게 모든 도구를 주면 어떻게 될까요? 각 Worker가 자기 도구만 쓸까요? 역할 밖 도구 호출은 몇 번일까요? 입력 토큰은 Context 분리 때보다 늘까요? 숫자로 적어 보세요.

In [ ]:
# 실패 재현: 역할 이름만 다르고 도구는 모두 같은 Worker 셋을 실행합니다.
announce("Persona만 나눈 Worker", model_calls=20)
role_log = defaultdict(lambda: {"모델 호출": 0, "입력 토큰": 0, "도구": []})

def role_meter(role):
    @wrap_model_call
    def metered_model(request, handler):
        response = handler(request)
        usage = next((m.usage_metadata for m in response.result if getattr(m, "usage_metadata", None)), None) or {}
        role_log[role]["모델 호출"] += 1
        role_log[role]["입력 토큰"] += usage.get("input_tokens", 0)
        return response

    @wrap_tool_call
    def metered_tool(request, handler):
        role_log[role]["도구"].append(request.tool_call["name"])
        return handler(request)
    return [metered_model, metered_tool]

def build_persona(desk):
    specs = [{**subagent_spec(role, desk, model), "tools": desk.tools("all"),
              "middleware": [*allowlist_middleware(desk.budget, ALL_TOOLS), *role_meter(role)]} for role in ("billing", "policy", "outage")]
    prompt = (f"고객 요청을 조사 Worker에게 task로 위임하고 결과로 최종 응답합니다. {scope_line(desk)} 서로 독립인 조사는 한 메시지에서 동시에 위임하세요.\n"
              f"금액 계산은 calculate_credit을 쓰세요.\n{OUTCOME_GUIDE}")
    return create_deep_agent(model=model, tools=desk.tools("writer"), subagents=specs, system_prompt=prompt, permissions=DENY_WRITES, name="supervisor-persona",
                             middleware=allowlist_middleware(desk.budget, frozenset({"task"}) | ROLE_TOOLS["writer"]), response_format=ProviderStrategy(FinalAnswer))

persona_row = run_custom("persona", double_pay, build_persona)
show_table([side_by_side(supervisor_row), side_by_side(persona_row)])

In [ ]:
def role_of_call(call):
    """모델 호출 때 보인 도구로 역할을 읽습니다. 모든 도구가 보이면 역할을 구분할 수 없습니다."""
    tools = set(call.get("tools") or [])
    if "task" in tools:
        return "supervisor"
    if not tools:
        return "router"
    matched = [role for role, names in ROLE_TOOLS.items() if tools <= names]
    return matched[0] if matched else ("fast" if tools == ROLE_TOOLS["policy"] | ROLE_TOOLS["billing"] else "전체 도구")

def split_roles(row):
    table = defaultdict(lambda: {"모델 호출": 0, "입력 토큰": 0, "도구": []})
    for call in row["calls"]:
        table[role_of_call(call)]["모델 호출"] += 1
        table[role_of_call(call)]["입력 토큰"] += call["input_tokens"] or 0
    for name in row["tools_called"]:
        owner = next((role for role, names in ROLE_TOOLS.items() if name in names), "supervisor")
        table["supervisor" if name in ("task", "calculate_credit") else owner]["도구"].append(name)
    return table

def role_rows(label, table):
    return [{"구조": label, "역할": role, **{k: entry[k] for k in ("모델 호출", "입력 토큰")}, "호출한 도구": dict(Counter(entry["도구"])),
             "역할 밖 도구 호출": sum(name not in ROLE_TOOLS.get(role, frozenset(entry["도구"])) for name in entry["도구"])} for role, entry in table.items()]

worker_tools = Counter(name for entry in role_log.values() for name in entry["도구"])
persona_table = {**role_log, "supervisor": {**split_roles(persona_row)["supervisor"], "도구": list((Counter(persona_row["tools_called"]) - worker_tools).elements())}}
show_table(role_rows("Context 분리", split_roles(supervisor_row)) + role_rows("Persona", persona_table))
print("거부된 도구 호출 Context 분리:", supervisor_row["denied"], "/ Persona:", persona_row["denied"], "| 모델 호출 0회")

In [ ]:
desk = make_desk(double_pay, label="shared")
specs = {role: subagent_spec(role, desk, model) for role in ("billing", "outage")}

def reached_desk(spec):
    """도구 함수가 잡고 있는 Desk를 꺼냅니다. 허용 목록과 무관하게 같은 프로세스의 같은 객체입니다."""
    return inspect.getclosurevars(spec["tools"][0].func).nonlocals["desk"]

show_table([{"Worker": role, "허용 도구": sorted(tool.name for tool in spec["tools"]), "같은 Desk": reached_desk(spec) is desk,
             "같은 원장": reached_desk(spec).ledger is desk.ledger} for role, spec in specs.items()])
reached_desk(specs["billing"]).ledger.refund("P-DEMO", 1, idempotency_key="shared-demo")
print("billing Worker 쪽에서 원장에 직접 쓴 뒤 outage Worker 쪽에서 본 원장 건수:", len(reached_desk(specs["outage"]).ledger.refunds))
print("이 실행의 임시 Desk에만 쓴 것입니다. 모델 호출 0회.")

### 그림으로 보기: 이름이 다른 것과 경계가 다른 것

아래 셀들이 비교하는 두 Worker 묶음입니다. 위는 이름만 다르고, 아래는 보이는 도구가 다릅니다. 두 묶음 모두 같은 `Desk`와 같은 원장에 닿습니다.

![Persona는 결제 담당, 규정 담당, 장애 담당 Worker가 모든 도구를 보고 같은 Desk와 원장에 닿는다. Context 분리는 billing, policy, outage Worker가 ROLE_TOOLS로 보이는 도구만 보지만 같은 Desk와 원장에 닿는다](./assets/05a-2-persona-vs-context.png)

- 줄어드는 것은 이름이 아니라 **보이는 도구와 넘기는 Context**입니다.
- 오른쪽 상자가 두 줄 모두 같다는 점이 마지막 셀의 `같은 Desk`, `같은 원장` 열입니다. Context 분리는 보안 격리가 아닙니다.

**관찰할 것.** 역할 이름이 줄이는 것과 도구 범위가 줄이는 것을 구분합니다.

- 역할 표에서 `Persona` 행의 `호출한 도구`와 `역할 밖 도구 호출`을 보세요. 0이라면 프롬프트와 위임 지시만으로 모델이 **이번에는** 자기 몫만 한 것입니다. 지켜졌다고 해서 보장된 것은 아닙니다. 다음 호출에서 어길 수 있고, 어겨도 막는 코드가 없습니다.
- 1 이상이면 역할 이름이 경계가 아니었다는 직접 증거입니다. 어떤 도구를 왜 불렀는지 함께 찾아보세요.
- `입력 토큰`을 역할별로 비교하세요. 도구 스키마가 늘면 Worker의 호출마다 입력이 함께 늡니다. 다만 `모델 호출`이 다른 행은 합계로 비교하지 마세요. 원문을 한 번 더 읽은 Worker(예: `read_evidence`를 부른 `policy`)는 도구 수와 상관없이 합계가 큽니다. 줄어드는 비용은 이름이 아니라 **보이는 도구와 넘기는 Context**를 줄여서 생깁니다.
- 마지막 셀의 `같은 Desk`, `같은 원장`이 True라는 것은 분리된 Worker도 같은 프로세스의 같은 객체에 닿을 수 있다는 뜻입니다. **Context 분리는 보안 격리가 아닙니다.** 모델에게 쓰기 도구를 주지 않은 것이 제한일 뿐, 코드 수준의 경계는 없습니다. 실제 격리가 필요하면 프로세스, 자격 증명, 원장 쪽의 권한으로 별도로 설계합니다. 이 과정에서 원장에 닿는 경로를 `gate_answer` 하나로 모은 이유도 여기 있습니다.

정리하면 나누는 기준은 "누구처럼 말하는가"가 아니라 "무엇을 보고, 무엇을 쓸 수 있는가"입니다. 그럼 절차를 재사용하고 싶을 뿐이라면 Worker까지 나눌 필요가 있을까요?

## 4. Skill vs SubAgent
**문제.** Worker를 만드는 이유가 "환불 절차를 매번 설명하기 싫어서"라면, 별도 실행 Context까지 필요하지 않을 수 있습니다. 절차는 Skill로 불러올 수 있고, 불러온 절차는 같은 Context 안에서 실행됩니다. SubAgent는 별도 실행 Context로 위임합니다. 두 가지는 해결하는 문제가 다릅니다.

Skill은 메타데이터(이름과 설명)가 항상 보이고, 본문은 Agent가 필요할 때 읽습니다. 그래서 메타데이터 크기는 모든 호출의 비용이고, 본문 크기는 읽었을 때의 비용입니다. 먼저 실제 크기를 봅니다.

02 3절의 자동으로 붙는 `general-purpose` Worker는 부모의 도구를 그대로 받으므로, 별도 실행 Context로 위임할 이유가 보이지 않는 SubAgent의 예입니다.

**먼저 예측하세요.** `single_skills`는 Single보다 모델 호출이 늘까요? 절차를 읽는 호출이 더해지는 대신 도구를 헤매는 호출이 줄 수도 있습니다. 어느 쪽이 클지, 자동 검사는 달라질지 적으세요.

In [ ]:
skill_rows = []
for path in sorted((ROOT / "skills").glob("*/SKILL.md")):
    _, front, body = path.read_text(encoding="utf-8").split("---", 2)
    skill_rows.append({"Skill": path.parent.name, "메타데이터 바이트": len(front.encode()), "본문 바이트": len(body.encode()),
                       "description 앞부분": next(line for line in front.splitlines() if line.startswith("description:"))[13:60]})
show_table(skill_rows)
print("메타데이터는 항상 입력에 들어가고, 본문은 읽을 때만 들어갑니다. 모델 호출 0회.")

In [ ]:
announce("single_skills로 같은 사례", model_calls=12)
skills_row = run_case("single_skills", double_pay, model=model)
show_table([skills_row], columns=RUN_COLUMNS)
print("읽은 Skill:", skills_row["loaded_skills"] or "없음")
print("부른 도구:", dict(Counter(skills_row["tools_called"])))

In [ ]:
show_table([side_by_side(row) for row in (single_rows[0], skills_row, supervisor_row)])
show_table([
    {"기준": "절차 재사용", "Skill": "적합. 본문을 필요할 때 읽습니다", "SubAgent": "가능하지만 과합니다", "코드": "판단이 없는 절차는 코드가 낫습니다"},
    {"기준": "Context 분리", "Skill": "없음. 읽은 본문과 조회 결과가 같은 Context에 쌓입니다", "SubAgent": "있음. 부모는 마지막 메시지만 받습니다", "코드": "해당 없음"},
    {"기준": "도구·권한 분리", "Skill": "없음. 같은 Agent의 도구를 씁니다", "SubAgent": "허용 목록을 Worker마다 겁니다", "코드": "게이트처럼 강제할 수 있습니다"},
    {"기준": "병렬 필요", "Skill": "없음", "SubAgent": "한 메시지의 task 여러 개로 가능합니다", "코드": "Send로 코드가 정합니다"}])
print("판단표는 기준을 정리한 것입니다. 위 표의 수치가 어느 칸을 뒷받침하는지는 직접 대조하세요. 모델 호출 0회.")

### 그림으로 보기: 코드, Skill, SubAgent를 고르는 기준

아래 판단표를 그림으로 옮겼습니다. 열이 쓸 수단이고 행이 따질 기준입니다.

![절차 재사용, Context 분리, 도구·권한 분리, 병렬 필요 네 기준에서 Skill, SubAgent, 코드를 비교한 표. Skill은 절차 재사용에 적합하고 Context와 권한 분리는 없다. SubAgent는 Context 분리와 Worker별 허용 목록을 준다. 코드는 판단이 없는 절차와 게이트에 쓴다](./assets/05a-3-skill-subagent.png)

- 행을 하나 고르고 세 열을 가로로 읽으세요. 어느 칸이 내 사례의 문제와 같은지가 고르는 근거입니다.
- 표는 기준을 정리한 것이고 성능을 비교하지 않습니다. 위 실행 표의 수치가 어느 칸을 뒷받침하는지는 직접 대조하세요.

**관찰할 것.** `읽은 Skill`이 비어 있으면 절차를 읽지 않고 답한 것입니다. 이 경우 Single과의 차이를 Skill의 효과로 말할 수 없습니다. 읽었다면 세 행의 `호출`, `입력 토큰`, `자동 검사`를 비교해 절차를 불러오는 비용이 무엇을 바꿨는지 보세요.

`첫 호출 입력 토큰`의 차이는 아직 아무 Skill도 읽기 전의 비용입니다. 위 표의 `메타데이터 바이트`만으로 설명되는지 따져 보세요. 메타데이터보다 차이가 훨씬 크다면 파일 도구 설명과 Skill 안내문까지 매 호출 함께 실린 것입니다. 또 `입력 토큰`은 호출마다의 입력을 더한 값이라서, 앞에서 읽은 내용이 뒤의 모든 호출에 반복됩니다. 그래서 본문이 작아도 합계는 크게 늘 수 있습니다. "Skill은 필요할 때만 읽으니 싸다"는 말도 이 표로 확인한 뒤에 하세요.

Skill과 SubAgent는 양자택일이 아닙니다. Worker가 Skill을 읽을 수도 있습니다. 판단 기준은 다음과 같이 정리할 수 있습니다.

- 같은 절차가 반복되지만 조회한 사실을 한 Context에서 함께 봐야 한다면 Skill을 먼저 검토합니다.
- 원문이 커서 부모 Context에서 떼어 놓아야 하거나 Worker마다 도구를 달리해야 한다면 SubAgent를 검토합니다.
- 판단 없이 입력을 곱하는 일이나 정책 검증은 코드에 둡니다. `calculate_credit`과 `check_refund`가 그 예입니다.

한 가지 전이 질문을 생각해 보세요. (`Day03/Day03_01.png`의 투자 심사 판서에서 가져온 질문입니다.) 투자 심사 보고서를 쓰는 Agent를 설계한다면 **문체, 데이터를 바라보는 관점, 심사역의 암묵지**는 어디에 담아야 할까요? 문체와 체크리스트는 Skill에 적을 수 있지만, 말로 꺼내 본 적이 없는 판단은 Skill에 적을 수 없습니다. 어떤 부분은 코드, 어떤 부분은 Skill, 어떤 부분은 사람의 검토로 남는지 자기 말로 나눠 보세요.

## 직접 해보기

유료 셀은 첫 줄의 `RUN_EXERCISE = False`를 `True`로 바꿔야 돌아갑니다. 수강생이 바꿀 값은 그 아래 대문자 상수로 모아 두었습니다. 모델을 부르지 않는 셀은 그대로 실행됩니다. 평균의 함정, 자동 검사의 한계, 새 사례를 다루는 해보기는 05b에 있습니다.

### 해보기 1

(실제 호출) 한 사례를 3회 반복해 호출 수와 답의 변동을 봅니다. 앞서 본 구조 사이의 차이가 이 변동보다 큰지 비교해 보세요. 반복이 3회여도 통계 주장에는 부족하다는 점을 잊지 마세요.

In [ ]:
RUN_EXERCISE = False
STRUCTURE, CASE_ID, REPEATS = "hybrid", "complex-double-pay", 3
if not RUN_EXERCISE:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")
else:
    announce("한 사례 3회 반복", model_calls=48)
    repeat_rows = run_grid([STRUCTURE], [case_by_id[CASE_ID]], model=model, repeats=REPEATS)
    show_table([{"반복": row["repeat"], "상태": row["status"], "호출": row["model_calls"], "입력 토큰": row["input_tokens"], "시간": row["elapsed_seconds"],
                 "자동 검사": evaluate_row(row, gold)["자동 검사"], "누락 사실": evaluate_row(row, gold)["누락 사실"], "답 앞부분": row["answer"][:60]} for row in repeat_rows])

### 해보기 2

예산을 **낮춰** 어느 구조가 먼저 실패하고 사유가 무엇인지 봅니다. 올려서 실패를 감추는 실험이 아닙니다. 값을 올리면 아래 `assert`가 멈춥니다. 실행 전에 규모가 먼저 출력되므로 보고 나서 켤지 정하세요.

In [ ]:
RUN_EXERCISE = False
LOWER_BUDGET = {**FIXED_BUDGET, "max_model_calls": 6, "max_seconds": 60}
TARGETS = ["single", "supervisor", "hybrid"]
assert all(LOWER_BUDGET[key] <= FIXED_BUDGET[key] for key in FIXED_BUDGET), "예산을 올려서 실패를 감추지 않습니다."
print("낮춘 예산:", LOWER_BUDGET, "| 원래 예산:", FIXED_BUDGET)
print(f"켜면 {len(TARGETS)}구조 × 1사례, 모델 호출 약 {len(TARGETS) * LOWER_BUDGET['max_model_calls']}회입니다. Router 호출도 예산이 셉니다.")
if not RUN_EXERCISE:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")
else:
    announce("낮춘 예산으로 3구조", model_calls=24)
    lowered = [run_case(name, double_pay, model=model, budget=LOWER_BUDGET) for name in TARGETS]
    show_table([{"구조": row["structure"], "상태": row["status"], "호출": row["model_calls"], "시간": row["elapsed_seconds"], "사유": row["errors"][:1], "outcome": row["outcome"]} for row in lowered])

## 정리

- **순서가 먼저입니다.** Single의 실패를 네 칸으로 분류하고, 프롬프트·도구·검색으로 고칠 수 있는 것을 먼저 고칩니다. 분리의 근거는 고친 뒤에도 남는 실패나 비용입니다.
- **이득과 비용은 같은 행에 씁니다.** 호출, 토큰, 시간, 자동 검사, 조정 바이트를 함께 읽습니다. 품질이 같다면 추가 비용이 무엇을 샀는지 답해야 합니다.
- **나누는 기준은 Persona가 아니라 Context 경계입니다.** 역할 이름은 도구 호출을 막지 못합니다. Context 분리 역시 보안 격리가 아니므로 권한은 따로 설계합니다.
- **절차는 Skill, 별도 실행 Context는 SubAgent, 판단이 없는 것은 코드**입니다. 조합할 수 있습니다.

05b에서는 이 구조들을 같은 조건의 표 하나로 모읍니다. 수치를 읽는 법, 지연을 가장 긴 경로로 읽는 법, Fast/Slow Path, 평균의 함정, 선택 기록을 다룹니다.

## 확인 질문

1. Multi-Agent로 바꾸기 전에 고쳐야 할 실패와 분리의 근거가 되는 실패를 자기 표에서 하나씩 들어 보세요.
2. 조정 비용은 어느 칸에 나타나나요? 품질이 같을 때 무엇을 근거로 비용을 정당화하나요?
3. "Context 분리는 곧 격리"라는 주장이 틀린 이유를 3절의 `같은 Desk`, `같은 원장` 출력으로 설명해 보세요.

다음 05b에서는 이 노트북에서 따로 본 구조들을 같은 조건의 표로 모읍니다. 05b는 이 노트북의 kernel 없이 돌도록 필요한 몇 가지 실행을 준비 절에서 다시 합니다.